In [1]:
import pandas as pd
from pathlib import Path

raw_file = Path(
    r"C:\Users\rajia\Downloads\FOIA_7a_FY2020_Present_asof_260630.csv"
)

In [2]:
keep_columns = [
    "AsOfDate", "Program", "LocationID",
    "BorrCity", "BorrState", "BorrZip",
    "BankName", "BankState",
    "GrossApproval", "SBAGuaranteedApproval",
    "ApprovalDate", "ApprovalFY",
    "ProcessingMethod", "InitialInterestRate",
    "FixedorVariableInterestInd", "TermInMonths",
    "NaicsCode", "NaicsDescription",
    "ProjectCounty", "ProjectState",
    "SBADistrictOffice", "CongressionalDistrict",
    "BusinessType", "BusinessAge",
    "LoanStatus", "PaidInFullDate", "ChargeOffDate",
    "GrossChargeOffAmount", "JobsSupported",
    "CollateralInd", "SoldSecMrktInd"
]

text_columns = {
    "LocationID": "string",
    "BorrZip": "string",
    "NaicsCode": "string",
    "CongressionalDistrict": "string"
}

df = pd.read_csv(
    raw_file,
    usecols=keep_columns,
    dtype=text_columns,
    low_memory=False
)

print(df.shape)
df.head()

(388338, 31)


,AsOfDate,Program,LocationID,BorrCity,BorrState,BorrZip,BankName,BankState,GrossApproval,SBAGuaranteedApproval,...,CongressionalDistrict,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,JobsSupported,CollateralInd,SoldSecMrktInd
0,2026-06-30,7A,0029805,Norristown,PA,19403,"TD Bank, National Association",DE,25000.0,12500.0,...,04,PARTNERSHIP,Unanswered,P I F,2024-11-30,NaN,0.0,5.0,N,NaN
1,2026-06-30,7A,0084894,ASTORIA,NY,11102,"Santander Bank, National Association",DE,95800.0,47900.0,...,10,CORPORATION,Existing or more than 2 years old,P I F,2023-11-30,NaN,0.0,3.0,N,NaN
2,2026-06-30,7A,0053803,COLFAX,CA,95713,"U.S. Bank, National Association",OH,40000.0,20000.0,...,01,INDIVIDUAL,Unanswered,P I F,2024-11-30,NaN,0.0,0.0,Y,NaN
3,2026-06-30,7A,0123499,CIBOLO,TX,78108,BayFirst National Bank,FL,125000.0,106250.0,...,15,CORPORATION,Existing or more than 2 years old,EXEMPT,NaN,NaN,0.0,3.0,N,Y
4,2026-06-30,7A,0070400,HOUSTON,TX,77006,Golden Bank National Association,TX,250000.0,187500.0,...,02,CORPORATION,Unanswered,P I F,2024-01-31,NaN,0.0,4.0,Y,NaN


In [3]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 481


In [4]:
print("\nLoan status:")
print(df["LoanStatus"].value_counts())

print("\nApproval fiscal years:")
print(df["ApprovalFY"].value_counts().sort_index())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))


Loan status:
LoanStatus
EXEMPT    242061
P I F      68201
CANCLD     50104
COMMIT     21079
CHGOFF      6893
Name: count, dtype: int64

Approval fiscal years:
ApprovalFY
2020    42298
2021    51856
2022    47678
2023    57362
2024    70242
2025    78078
2026    40824
Name: count, dtype: int64

Missing values:
ChargeOffDate                 381450
PaidInFullDate                320136
SoldSecMrktInd                271574
BusinessAge                      186
BusinessType                      34
CongressionalDistrict             24
InitialInterestRate                8
FixedorVariableInterestInd         8
NaicsDescription                   0
CollateralInd                      0
dtype: int64


In [5]:
print(df["LoanStatus"])

0          P I F
1          P I F
2          P I F
3         EXEMPT
4          P I F
           ...  
388333    COMMIT
388334    COMMIT
388335    COMMIT
388336    COMMIT
388337    COMMIT
Name: LoanStatus, Length: 388338, dtype: object


In [6]:
# Remove extra spaces from text fields
for column in df.select_dtypes(include=["object", "string"]).columns:
    df[column] = df[column].str.strip()

# Convert date columns to proper dates
date_columns = [
    "AsOfDate", "ApprovalDate",
    "PaidInFullDate", "ChargeOffDate"
]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

# Convert financial and numeric columns
numeric_columns = [
    "GrossApproval", "SBAGuaranteedApproval",
    "InitialInterestRate", "TermInMonths",
    "GrossChargeOffAmount", "JobsSupported"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["ApprovalFY"] = pd.to_numeric(
    df["ApprovalFY"], errors="coerce"
).astype("Int64")

In [7]:
starting_rows = len(df)

df = df.drop_duplicates().copy()

print("Rows before:", starting_rows)
print("Rows after:", len(df))
print("Duplicates removed:", starting_rows - len(df))

Rows before: 388338
Rows after: 387857
Duplicates removed: 481


In [8]:
# This is a new technical ID because the source does not provide a unique loan ID.
df.insert(0, "loan_record_id", range(1, len(df) + 1))

df["approval_year"] = df["ApprovalDate"].dt.year
df["approval_month"] = df["ApprovalDate"].dt.month
df["approval_quarter"] = df["ApprovalDate"].dt.quarter

df["guarantee_pct"] = (
    df["SBAGuaranteedApproval"] / df["GrossApproval"]
)

df["charge_off_flag"] = (
    df["LoanStatus"] == "CHGOFF"
).astype(int)

df["paid_in_full_flag"] = (
    df["LoanStatus"] == "P I F"
).astype(int)

df[[
    "GrossApproval", "SBAGuaranteedApproval",
    "guarantee_pct", "charge_off_flag", "paid_in_full_flag"
]].head()

,GrossApproval,SBAGuaranteedApproval,guarantee_pct,charge_off_flag,paid_in_full_flag
0,25000.0,12500.0,0.50,0,1
1,95800.0,47900.0,0.50,0,1
2,40000.0,20000.0,0.50,0,1
3,125000.0,106250.0,0.85,0,0
4,250000.0,187500.0,0.75,0,1


In [9]:
df_complete_years = df[
    df["ApprovalFY"].between(2020, 2025)
].copy()

print(df_complete_years["ApprovalFY"].value_counts().sort_index())

ApprovalFY
2020    42191
2021    51737
2022    47609
2023    57317
2024    70199
2025    78009
Name: count, dtype: Int64
